# model_a5 — deneme 2

**GERI BESLEMELI AGIRLIK ORTALAMASI (Lookahead).  TEK FARK: ort_bas 0 -> 10000.**
Taban `model_a4` (wd=0.5, 2x veri). 10.000'den sonra HER ESIKTE:

```
phi   <- (1 - alfa) * phi + alfa * theta        alfa = 0.5
theta <- phi        egitim ORTALANMIS agirliktan DEVAM EDER
```

Golge ortalama DEGIL — o zaten `pencere_a`nin kosudan sonra yaptigi sey.

Onceden kayit `belge/onkayit/model_a5.md` — **kosudan once yazildi**.

### Bunun adi var: Lookahead (1907.08610, NeurIPS 2019)

Zhang, Lucas, Ba, Hinton. `alfa=0.5` tam olarak "bir onceki modelle esit
ortalama". **TEK FARK OLCEK:** makale k=5..10 adim tariyor, burada
k=`olc_her`=2000 — **200-400 KATI**. Lookahead'in taradigi bolge DEGIL;
SWA'ya yakin ama SWA geri besleme yapmaz.

> ⚠️ **RISK ACIK:** alfa=0.5 ile 2000 adimda bir ortalamak ogrenmeyi
> SONUMLEYEBILIR. Cikarsa once `ort_her` sorgulanmali, fikir degil.

### Neden bu projede anlamli

```
                 en iyi TEK       SON pencere
                 anlik goruntu    (agirlik ort.)    oran
model_a1            0.2290           0.3473        1,52x
model_a2            0.2693           0.4220        1,57x
model_a             0.2767           0.4947        1,79x
model_a3            0.3830           0.8743        2,28x
```

Kazanc buyuk ve `wd` ile buyuyor — ama su ana kadar hep KOSUDAN SONRA
alindi. Soru: egitimin ICINE konursa birikir mi?

### Kiyas neye karsi

```
model_a4  52000-60000   ent 0.8820   comp 0.9900   ent_yok 0.5057

ent >= 0.93      ->  GERI BESLEME KAZANDIRIYOR
ent 0.83 - 0.93  ->  AYIRT EDILEMEDI
ent <  0.83      ->  ZARAR VERIYOR (muhtemelen sonumleme)
```

### Asil teshis: egri/pencere ORANI

```
model_a4: egri 0.4923 -> pencere 0.8820   (1,79x)
Geri besleme calisiyorsa bu oran KUCULMELI -- egitim ortalamayi zaten
yapmis olur. Oran kuculurken PENCERE korunuyorsa kazanc iceri alindi;
pencere de dusuyorsa kazanc kayboldu.
```

---

### Her hucrenin basligindaki etiket ne demek

> **tekrar:** guvenli · **GPU:** hayir · **yazar:** hayir

| alan | ne sorar |
|---|---|
| **tekrar** | Bu hucreyi ikinci kez calistirmak guvenli mi? |
| **GPU** | GPU kullanir mi — yani kosan egitimi yavaslatir mi? |
| **yazar** | Drive'a bir sey yazar mi? |

**Sira:** 0 → 1 → 2 → 3 → 4 ile baslat. Kosu surerken **5 ve 6**.
Bitince **7**. **3'u kosu surerken calistirma**, 8 kosuyu oldurur.

## 0 — Model adı ve yollar

Değiştirilecek **tek satır** burada: `MODEL`. Depo yolu ve Drive yolu
ondan türer — ikisi de elle yazılmaz.

**Log adı burada üretilmez**, onu 4. hücre her başlatmada kendisi basar.
Böylece iki koşu aynı log adını alamaz, hücre sırası ne olursa olsun.

> **tekrar:** güvenli, her zaman · **GPU:** hayır · **yazar:** hayır

In [ ]:
MODEL = "model_a5"               # <-- DEGISTIRILECEK TEK SATIR, orn. "model_a"

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
# LOG ADI BURADA URETILMEZ -- 4. hucre her BASLATMADA kendi damgasini
# basar. Sebep olculdu (15 Eylul): log adi burada uretilince, bu hucreyi
# yeniden calistirmadan ikinci bir kosu baslatmak ONCEKI kosunun logunu
# "w" ile ACIP SIFIRLIYOR. Fiilen oldu: 20.000'lik kosunun logu, 40.000'lik
# kosu baslayinca silindi. Damga BASLATAN hucrede uretilirse imkansiz.
print(MODEL, "->", EV)

## 1 — GPU var mı, boş mu

Hız ÖLÇÜLDÜ (15 Eylül, T4, t0). Ardışık iki ölçüm noktasının
FARKINDAN, ortanca değer:

```
model_a   l=4 dongu=2   32,3 ms/adim   20.000 adim  10,8 dk
model_a1  l=4 dongu=1   18,2 ms/adim   20.000 adim   6,1 dk
model_a2  l=8 dongu=1   34,1 ms/adim   20.000 adim  11,4 dk
```

Döngü adım başına **1,77×** pahalı — hesabın neredeyse tamamı
bloklarda. Sekiz AYRI katman (`model_a2`) ise döngülü sekiz
katman-eşdeğerinden yalnız **1,06×** pahalı: aynı iş, farklı ağırlıklar.

> ⚠️ **`egri`deki `sn` HER SÜRDÜRMEDE SIFIRLANIR.**
> `sn[20000]/20000` hesabı YANLIŞTIR — 15 Eylül'de tam böyle yanıldım:
> `model_a`nın 20.000. adımı 8.000'de başlamış bir oturumun içindeydi,
> 22,3 ms çıktı (gerçek 32,3) ve bu yanlış sayı dört deftere birden
> yazıldı. Adım başına süre **ardışık iki ölçüm noktasının
> farkından** hesaplanır.

CPU'ya düşerse aynı iş **saatler** sürer — arşivde adım başına 1,64 sn
ölçülmüştü, bu da 20.000 adım için ~9 saat eder. Burada durmak, 9 saat
sonra fark etmekten iyidir.

> **tekrar:** güvenli · **GPU:** sorar, kullanmaz · **yazar:** hayır

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

## 2 — Drive

Çıktı doğrudan Drive'a yazılır. `/content` runtime ölünce silinir, Drive silinmez.

`drive.mount` çalışmazsa `/content/drive/MyDrive/...` **sihirli bir yol
değildir** — sıradan bir klasördür ve `os.makedirs` onu geçici diskte
sessizce açar. Koşu biter, runtime ölür, her şey silinir.
`ismount` bunu ayırır.

> **tekrar:** güvenli · **GPU:** hayır · **yazar:** evet — yalnızca klasör
> açar (`<model>/log/`), hiçbir koşu dosyasına dokunmaz

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

## 3 — Kodu GitHub'dan çek

**Colab'da yama yok.** Depo silinip yeniden klonlanır, üzerine hiçbir şey
yazılmaz. Tek kaynak: GitHub.

Yereli düzeltip itmeyi unuttuysan burada görürsün — klon eski commit'i getirir.

Bu hucre ayrica ailenin **kilit testini** (`test_*.py`) kosturur:
taban `model_a` degismedi mi, kollarin dugme yapisi bozulmadi mi.
Duserse egitim **baslamaz** -- kayitli sonuclar o tabana dayaniyor.

> **tekrar:** ⚠️ **EĞİTİM KOŞARKEN ÇALIŞTIRMA.** İlk işi `rm -rf /content/kod`
> ve koşan süreç tam o klasörden import etmiş durumda. Koşu yokken güvenli.
> · **GPU:** hayır · **yazar:** `/content` (Drive'a **değil**)

In [ ]:
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
_aday = glob.glob(f"{KOD}/deneme2/*/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

# --- IMPORT ONBELLEGINI TEMIZLE -- BU HUCRENIN EN SESSIZ TUZAGI ----------
# `rm -rf` + yeniden klon KODU tazeler ama `sys.modules` ESKI modul
# nesnesini tutar. Ayni cekirdekte MODEL degistirip bu hucreyi yeniden
# kosarsan, yeni kodu klonlamis ama ESKI modulu kullaniyor olursun.
# OLCULDU (15 Eylul): model_a4'ten model_a5'e gecerken
#   "AssertionError: Ayar'da boyle alan yok: {'ort_bas'}"
# cikti -- cunku model_a hala onceki klondan gelen, `ort_bas`i olmayan
# nesneydi. Daha sinsi hali: alan adlari tutarsa hata VERMEZ ve kosu
# ESKI KODLA baslar; kunyedeki commit ise YENIYI gosterir.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", sorted(_atilan))

# --- TABAN DEGISMEDI MI --------------------------------------------------
# Ailenin butun kollari ayni tabani (model_a.py) import ediyor ve KAYITLI
# sonuclar o tabana dayaniyor. `test_sabit.py` tabanin AYAR'ini, grafi,
# gordugu veriyi, parametre sayisini ve ailenin DUGME YAPISINI kilitler.
# Duserse egitim BASLAMAMALI.
#
# CIKTI YAKALANIR ve BASILIR: Colab alt surec stdout'unu hucreye
# aktarmiyor; "cikti yok" ile "test kosmadi" ayrimi sansa birakilmaz.
# AYRINTI=1 -> gecen kontroller de basilir (dugme tablosu gorunur olsun).
_t = [x for x in glob.glob(f"{AILE}/test_*.py")]
if _t:
    print()
    print("=" * 72)
    _r = subprocess.run([sys.executable, _t[0]], cwd=AILE,
                        capture_output=True, text=True,
                        env={**os.environ, "AYRINTI": "1",
                             "KOSU_KOK": EV.rsplit("/", 1)[0]})
    print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
    if _r.stderr.strip():
        print("stderr:", _r.stderr.rstrip()[-2000:])
    print("=" * 72)
    assert _r.returncode == 0, (
        f"{os.path.basename(_t[0])} DUSTU (cikis {_r.returncode}). Taban "
        f"degismis olabilir ve KAYITLI sonuclar ona dayaniyor. EGITIM BASLATMA.")
    print()

sys.path.insert(0, AILE)
sys.path.insert(0, os.path.dirname(AILE))     # veri modulleri icin
M = importlib.import_module(MODEL)
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
# Yuklenen modul GERCEKTEN yeni klondan mi geldi?
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi: {M.__file__}"
print("ayar:", M.AYAR)

## 4 — Başlat

**Önce tek tohum** (`t0`). Sonuç olumluysa yeter; olumsuzsa `TOHUMLAR`'ı
`[1, 2]` yapıp tekrar koşarsın — `t0` klasörüne dokunulmaz.

Çıktı Drive'da, **tepe klasör modelin adı**:

```
MyDrive/<model>/
    OKU.md        klasörü anlatır, her koşuda yenilenir
    log/          kos_<zaman>.txt
    t0/           ayar_ · kosu_ · egri_ · pencere_ json'ları
        snap/     snap_<model>_t0_00020000.pt
```

Dosya adları da tohumu taşır — klasörden çıkarmak gerekmiyor.

**Bütçeyi uzatmak** (CLAUDE.md kural 1): `ADIM`'ı büyüt **ve**
`SURDUR = True` yap. Uzatma sıfırdan koşu değil, **sürdürmedir** — koşu
kaldığı yerden devam eder. 20.000 bir tavan değil, ilk sınır.

> **tekrar:** ⛔ **hayır** (yeni koşu için). Üç koruma var: hücre, süreç
> hâlâ koşuyorsa ikincisini başlatmaz; `kos.py` dolu tohum klasörünü
> reddeder; sürdürme paketi yokken `SURDUR = True` koşuyu durdurur.
> `SURDUR = True` ile tekrar çalıştırmak ise **kasıtlı ve güvenlidir**.
> · **GPU:** evet (alt süreç; ölçüldü: model_a 20.000 adım **10,8 dk**,
> model_a2 **11,4 dk**, model_a1 **6,1 dk**)
> — hücre kendi içinde **GPU kapısını** sorar (kural 2)
> · **yazar:** Drive

**Ayrı süreç.** Çekirdek serbest kalır: ilerlemeye bakabilir, durdurabilirsin.

Koşan kod `deneme2/kos.py` — **depoda**, defterin içinde metin olarak
kurulmuyor. Yani başlatan kod da koşan kodla aynı commit'te.

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

TOHUMLAR = [0]             # ONCE TEK TOHUM.
# Sonuc OLUMLU cikarsa (ent yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
#
# Neden onemli: grokking tohuma bagli (2603.25009'da "only 1 of 3 seeds
# grokked" gibi sonuclar var). Tek tohumda ent yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

ADIM   = None    # None = model_a.py'deki ILK SINIR (20.000).
#                  Uzatmak KULLANICI karari (CLAUDE.md kural 1). Yazili bir
#                  UST SINIR YOK; 40.000 da 60.000 da denebilir.
#                  UZATMA = SURDURME: asagidaki SURDUR'u da True yap.
SURDUR = False   # True = surdurme_t<N>.pt'den KALDIGI YERDEN devam.
#                  Paket YOKSA kosu DURUR ve bunu soyler -- sessizce bastan
#                  baslamaz. Surdurme destegi 15 Eylul'de eklendi; ondan
#                  onceki kosularda paket YOK.
USTUNE = False   # True = dolu klasoru t<N>_eski_<zaman>/'a TASI (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

_arg = [sys.executable, "-u", f"{KOD}/deneme2/kos.py",
        "--model", MODEL, "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
# LOG ADI HER BASLATMADA YENI: bu hucre iki kez calisirsa iki AYRI log
# olur, oncekinin uzerine YAZILMAZ.
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")
print("Ayni tohumu bilerek tekrar kosmak icin --ustune; o da SILMEZ,")
print("eskisini t<N>_eski_<zaman>/ diye yan klasore TASIR.")

## 5 — İlerleme (ham log)

Alt sürecin çıktısının sonunu basar. **Hesap yapmaz, GPU kullanmaz** —
sadece dosya okur.

> ⚠️ **Log GERİDEN GELEBİLİR.** Alt sürecin `stdout`'u Drive'a yazılıyor ve
> FUSE katmanı onu tamponluyor. Ölçüldü (15 Eylül, canlı koşuda): log hâlâ
> adım 0'ı gösterirken 6. hücre adım 2000'i gösteriyordu. **Canlı durum için
> 6. hücreye bak**; bu hücre traceback okumak için.
>
> **tekrar:** güvenli, istediğin kadar · **GPU:** hayır · **yazar:** hayır
>
> Çekirdek yeniden başladıysa (`p` kaybolur) bu hücre yine çalışır: süreç
> tablosuna bakıp koşunun yaşayıp yaşamadığını söyler.

In [ ]:
import glob, subprocess
# LOG degiskenine DEGIL, klasordeki EN YENI log'a bak.
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"

# `p` CEKIRDEK YENIDEN BASLAYINCA KAYBOLUR -- ve tam o an bu hucreye
# ihtiyac duyulur. Olculdu (15 Eylul): Colab cekirdegi oldu, bu hucre
# `NameError: name 'p' is not defined` verdi, koşunun yasayip yasamadigi
# ogrenilemedi. Artik `p` yoksa SUREC TABLOSUNA bakiyor.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("!! `p` YOK -- cekirdek yeniden baslamis.")
    if _ps:
        print("   ama SUREC YASIYOR:\n   " + _ps)
    else:
        print("   ve kos.py sureci de YOK -> kosu OLDU.")
        print("   Drive'i yeniden bagla (2. hucre), 3'u kos, sonra 4. hucrede")
        print("   SURDUR = True ile KALDIGI YERDEN devam ettir.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log dosyasi var)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

## 6 — Rapor (canlı durum)

Eğri ve künye dosyalarını Drive'dan okur; ikisi de **atomik** yazıldığı için
yarım dosya okunmaz ve **5. hücrenin aksine geriden gelmez**.

Koşu sürerken bakılacak hücre budur: hangi adımda, hangi kapı geçti,
`durum` ne.

> Yine de bu bir **ön okuma**. Birincil okuma `pencere_a.py` (7. hücre) —
> o, N anlık görüntünün **ağırlık ortalamasını** alıp tek model ölçer. Eğri
> değerleriyle pencere değerleri **aynı şey değildir**; arşivde ikisi ~2×
> farklı çıktı.
>
> **tekrar:** güvenli, istediğin kadar · **GPU:** hayır · **yazar:** hayır

In [ ]:
import json, glob, os

# `ent_yok` DOGRULUGU da olculuyordu ama tabloda YOKTU (15 Eylul): egri
# json'ina yaziliyor, rapora girmiyordu. Olculup gosterilmeyen sayi, yok
# sayilan sayidir -- eklendi.
# SUTUNLAR EGRININ KENDISINDEN TURETILIR, elle yazilmaz. `ent_kati`
# (kati_pay > 0) ancak boyle gorunur; sabit liste olsaydi olculur ama
# BASILMAZDI. Bu tam olarak `ent_yok`un basina gelen seydi (15 Eylul):
# egri json'ina yaziliyordu, rapora girmiyordu.
_TUM = (("adim", "adim"), ("kayip", "kayip"), ("one", "one"),
        ("seen", "seen"), ("comp", "comp"), ("ent", "ent"),
        ("ent_kati", "ent_kati"), ("ent_yok", "ent_yok"),
        ("ent_kisayol", "ent_ksy"), ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{os.path.basename(kl)}: egri YOK"); continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    _var = set().union(*(set(r) for r in e))
    SUT = tuple(x for x in _TUM if x[0] in _var)
    print("=" * 84)
    print(f"{os.path.basename(kl)}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    v = k.get("veri", {})
    if v:
        print(f"   olgu {v['olgu']}  egitim2 {v['egitim2']}  ENT {v['ent']}  "
              f"phi {v['phi']} (wang {v['wang_phi']})  "
              f"parametre {k.get('parametre',0):,}  iz {k.get('olcme_izi','?')}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 81)
    for ad, kural, g in (
            ("SAGLIK-1HOP",  "one >= 0.98",  s.get("one", 0) >= 0.98),
            ("SAGLIK-EZBER", "seen >= 0.95", s.get("seen", 0) >= 0.95),
            ("OLGUNLUK",     "comp >= 0.50", s.get("comp", 0) >= 0.50),
            ("BIRIM TESTI",  "ent_yok_kisayol == 0",
             abs(s.get("ent_yok_kisayol", 1)) < 1e-9)):
        print(f"   {'GECTI ' if g else '!! KALDI'}  {ad:<14} {kural}")
    if len(e) >= 2:
        print(f"   ent son iki olcumde {e[-1].get('ent',0)-e[-2].get('ent',0):+.4f}"
              "   (BUTCE hukmu pencere_a'da verilir, burada DEGIL)")

## 7 — `pencere_a` · BİRİNCİL OKUMA

N anlık görüntünün **ağırlık ortalamasını** alır, sonra **tek** model ölçer.
Eğri değerlerinin ortalaması değil — ikisi ayrı şey.

> **tekrar:** güvenli; dosya adı genişliği taşır
> (`pencere_<model>_t0_g5.json`), farklı genişlikler birbirini ezmez.
> · **GPU:** ⛔ **KOŞU SÜRERKEN ÇALIŞTIRMA.** Hücre kendi içinde
> `assert _bos > 2.0` soruyor; eğitim süreci belleği tuttuğu için bu
> kontrol DÜŞER ve hücre hata verir. Ölçüldü: kullanıcı 16 Eylül'de
> denedi, hata aldı. **Koşu BITTIKTEN sonra** çalıştır.
> · **yazar:** Drive (tohum klasörüne bir json)
> — hücre kendi içinde **GPU kapısını** sorar (kural 2)

### Eğitim koşarken hangi hücre çalıştırılabilir

| hücre | koşu sürerken | neden |
|---|---|---|
| 0 Ayarlar | ✅ | sadece değişken |
| 1 GPU | ✅ | sorar, kullanmaz |
| 2 Drive | ✅ | klasör açar |
| 3 Kod | ⛔ | `rm -rf /content/kod` — koşan süreç oradan import etti |
| 4 Başlat | ⛔ | hücre zaten engelliyor |
| 5 İlerleme | ✅ | dosya okur (geriden gelebilir) |
| 6 Rapor | ✅ | dosya okur, **güncel** |
| 7 `pencere_a` | ⚠️ | çalışır ama GPU'yu paylaşır, eğitim yavaşlar |
| 8 Durdur | ⛔ | koşuyu öldürür |

5 ve 6'nın güvenli olmasının sebebi teknik: anlık görüntü ve eğri **atomik**
yazılıyor (`.tmp` → `os.replace`), yani okuyucu ya eski ya yeni dosyayı
görür, arasını asla. Ölçüldü: atomik olmadan yarım bir `.pt`
`torch.load`'ı patlatıyor **ve** `pencere_a`'nın glob'una giriyordu.

**İKİNCİ DEFTER = İKİNCİ RUNTIME = İKİNCİ GPU.** Colab'da yeni defter açmak
aynı GPU'yu paylaşmaz. Orada sadece **2 → 3 → 7**'yi çalıştır, **4'e
dokunma**. Eğitim hiç etkilenmez; bedeli aynı anda iki Colab oturumu.

In [ ]:
# --- GPU KAPISI (CLAUDE.md kural 2) -------------------------------------
# GPU kullanacak her hucre, ONCE kendi icinde GPU'yu sorar. Ayri bir
# "GPU var mi" hucresi HUCRE SIRASINA bagli bir kuraldir; icerideki
# kontrol degildir. Olculdu (15 Eylul): Colab cekirdegi kosu sirasinda
# oldu, GPU ve Drive durumu gitti; ayri kontrol hucresi vardi ama
# calistirilmamisti.
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")
# ------------------------------------------------------------------------

# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

## 8 — Durdurmak, ve tekrar koşmak

`p.kill()` — ya da runtime'ı kapat. Anlık görüntüler Drive'da kalır,
`kosu_t<N>.json`'daki `durum` `HATA` olur ve rapor koşunun **bitmediğini**
söyler.

### Sürdürme VAR (15 Eylül'den beri)

Her ölçüm noktasında tek bir `surdurme_t<N>.pt` atomik olarak yazılır:
fp32 ağırlık **+ AdamW momentleri + GradScaler ölçeği + batch RNG durumu**.
Koşu koparsa en fazla `olc_her` adım kaybedilir.

Devam etmek ya da bütçeyi uzatmak için 4. hücrede:

```python
ADIM   = 40000     # yeni tavan
SURDUR = True      # kaldigi yerden
```

Sınandı: 8+8 adım sürdürülmüş koşu, kesintisiz 16 adımlık koşuyla **bit
düzeyinde aynı** çıktı (ağırlık farkı `0.000e+00`). Anlık görüntüler bunu
yapamaz — içlerinde yalnız fp16 ağırlık var, optimizer durumu **yok**;
oradan devam etmek sessizce **başka bir yörünge** üretirdi.

> Sürdürme desteği eklenmeden **önce** koşulmuş klasörlerde paket yoktur.
> `SURDUR = True` o durumda koşuyu **durdurur** ve bunu söyler; sessizce
> baştan başlamaz.

### Tekrar koşunca ne kaybolur? — hiçbir şey

```
BASKA TOHUM (t1, t2)   ayri klasore yazar, t0'a DOKUNMAZ
AYNI TOHUM             kosu REDDEDILIR: "t0 ZATEN DOLU"
AYNI TOHUM + --ustune  eskisi SILINMEZ, t0_eski_<zaman>/ diye TASINIR
```

Üçü de ölçülerek sınandı. Eskiden `--ustune` klasörü temizlemeden üstüne
yazıyordu: 8 adımlık koşunun üstüne 4 adımlık koşu gelince `snap/` içinde
**iki koşunun anlık görüntüleri yan yana** kalıyordu ve `pencere_a` bunları
tek pencerede ortalıyordu — sessizce.

> **tekrar:** ⛔ koşuyu **öldürür**. Satırın başındaki `#` bilerek duruyor.
> · **GPU:** hayır · **yazar:** hayır

In [ ]:
# p.kill()